In [0]:
fetch_date = dbutils.widgets.get("fetch_date")
source_table = dbutils.widgets.get("source_table")
landing_table = dbutils.widgets.get("landing_table")
office_table = dbutils.widgets.get("office_table")
client_table = dbutils.widgets.get("client_table")

In [0]:
display(
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW atb_src AS
SELECT 
  CAST(ReportingDate AS DATE) AS ReportingDate, 
  CAST(FacilityCode AS INT) AS FacilityCode, 
  CAST(AcctNbr AS STRING) AS AcctNbr, 
  CAST(AcctBalance AS DOUBLE) AS AcctBalance, 
  CAST(SourceSystemKey AS INT) AS SourceSystemKey,
  current_timestamp() AS _load_timestamp
FROM (
  WITH 
  atb_cte AS (
    SELECT  
      to_date(CAST(date_entered_key AS STRING), 'yyyyMMdd') AS ReportingDate,
      CAST(ofc.OfficeNumber AS INT) AS FacilityCode, 
      CASE UPPER(Invoice_Number)
        WHEN 'ADV' 
          THEN CONCAT('ADV', ' - ', clt.SourceSystemId) 
        ELSE Invoice_Number
      END AS AcctNbr, 
      Account_Balance AS AcctBalance, 
      0 AS SourceSystemKey
    FROM {source_table} obd
    LEFT JOIN {office_table} ofc
      ON obd.office_Key = ofc.Officekey
    LEFT JOIN {client_table} clt
      ON clt.ClientKey = obd.client_key
    WHERE obd.account_balance != 0
    AND obd.date_entered_key = date_format(DATE('{fetch_date}'), 'yyyyMMdd')
  ),
  atb_clean AS (
    SELECT *,
    row_number() OVER (
      PARTITION BY AcctNbr, FacilityCode
      ORDER BY AcctNbr
    ) AS rn
    FROM atb_cte
  )
  SELECT ReportingDate, FacilityCode, AcctNbr, AcctBalance, SourceSystemKey
  FROM atb_clean
  WHERE rn=1
) AS src
""")
)

In [0]:
display(
spark.sql(f"""
MERGE INTO {landing_table} tgt
USING atb_src src
    ON tgt.AcctNbr = src.AcctNbr
    AND tgt.ReportingDate = src.ReportingDate
    AND tgt.SourceSystemKey = 0

WHEN MATCHED THEN
  UPDATE SET
    tgt.FacilityCode = src.FacilityCode,
    tgt.AcctBalance = src.AcctBalance,
    tgt._load_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
  INSERT (
    ReportingDate,
    FacilityCode,
    AcctNbr,
    AcctBalance,
    SourceSystemKey,
    _load_timestamp
  )
  VALUES (
    src.ReportingDate,
    src.FacilityCode,
    src.AcctNbr,
    src.AcctBalance,
    src.SourceSystemKey,
    current_timestamp()
  );
""")
)